## 📎 Code this notebook uses
*(how to run, and which code each step uses: [`README.md`](README.md))*

- **`code/pipeline/compute/`** (in this order): `certify_joint_numerics.py`,
  `compute_joint_fit_sensitivities.py`, `compute_joint_fit_checks.py`, `compute_joint_diviner.py`,
  `compute_gradient_epochs.py`, `compute_joint_chi_density.py`, `compute_likelihood_ratio.py`
- **`code/pipeline/figures/`:** `make_joint_figures.py` (Figure 3, SI Figure S6)

# 04 — Uncertainty and tests

How sure the result is, and how it compares with the global value: the solver
tests (SI Table S5), the error budget (Table 3), the model at the fit against
the sensors (Table 1), the Diviner comparison (SI Table S11), the $\chi$ and
density tests (SI Table S14) and the likelihood-ratio tests (Tables 1, 2).

In [ ]:
# How much to run (README.md explains the three modes):
#   "show"  : run nothing, read the stored results            (minutes)
#   "quick" : run the short steps and the fast form of long ones  (~30 min for all notebooks)
#   "full"  : run everything from scratch, as for the paper    (about 6 h for all notebooks)
MODE = "show"

import pandas as pd
from lunar import notebook as nb
nb.set_mode(MODE)
SITES_ = ("A15", "A17")

## 1. Solver tests (SI Table S5)

The reported $K_d^*$ found again with one numerical setting changed at a time:
time step, grid, anchor depth, run length, starting temperature, $K_d$ spacing.

In [ ]:
nb.step("compute/certify_joint_numerics.py", minutes=3, writes="joint_numerics_certification.json")
cert = nb.load("joint_numerics_certification.json")["sites"]
display(pd.DataFrame({s: {k: f"{v:+.5f}" for k, v in cert[s]["delta_kd_mW"].items()} for s in SITES_})
        .rename_axis("change of K_d* (mW m^-1 K^-1)"))

In [ ]:
chk = nb.Checks("Compared with the paper (SI Table S5):")
mx = lambda key: max(cert[s]["summary"][key] for s in SITES_)
chk.add("length of each shallow run", mx("n_inner_max"), "0.0002", ".4f")
chk.add("Step A on the full column", mx("full_column"), "0.0006", ".4f")
chk.add("time step halved", mx("dt_per_halving_max"), "0.001", ".3f")
chk.add("finer grid", mx("grid_fine"), "0.0024", ".4f")
chk.add("anchor depth", mx("anchor_max"), "0.0001", ".4f")
chk.add("K_d spacing", mx("kd_spacing"), "0.0003", ".4f")
chk.add("starting temperature: profiles change by about (K)", mx("start_profile_max_K"), "1e-5", ".0e")
chk.show()

## 2. The error budget and the model comparison (Tables 2, 3; SI Figure S6)

Each measured input changed over its range and the joint fit redone. Full run
about 80 min. "quick" reuses the stored runs when
`results/joint_fit_sensitivities_cache.npz` is on your computer (it is not on
GitHub; 27 MB); otherwise it shows the stored result.

In [ ]:
nb.step("compute/compute_joint_fit_sensitivities.py", minutes=80, writes="joint_fit_sensitivities.json",
        quick=("--reuse",), quick_needs="results/joint_fit_sensitivities_cache.npz")
eb = nb.load("joint_fit_sensitivities.json")["error_budget"]
rows = ["sigma_stat", "sigma_solver", "sigma_Qb", "sigma_rho", "sigma_cp", "sigma_Ks",
        "sigma_zb", "sigma_thr", "sigma_window", "sigma_epoch", "total_quadrature"]
display(pd.DataFrame({s: [round(eb[s][r], 4) for r in rows] for s in SITES_}, index=rows)
        .rename_axis("Table 3 (mW m^-1 K^-1)"))

## 3. The model at the fit, against the sensors (Table 1; Sections 3.3, 4.2)

The model solved once more at exactly the fitted albedo and $K_d$, and at the
global $K_d=3.4$ with its own fitted albedo, and the Martinez & Siegler (2021)
model. Also the hold-out of the deepest sensor and the TG/TR split.

In [ ]:
nb.step("compute/compute_joint_fit_checks.py", minutes=1, writes="joint_fit_checks.json")
fc = nb.load("joint_fit_checks.json")
rows = []
for s in SITES_:
    for key, name in (("fit_with_diffusivity", "Hayne, joint fit"), ("global_3p4", "Hayne, K_d = 3.4, A refitted"),
                      ("martinez_forward", "Martinez & Siegler")):
        r = fc[s][key]
        kap = sum(r["kappa_model"]) / 2 if "kappa_model" in r else None
        rows.append((s, name, round(r["A"], 3), round(r["rmse_K"], 3), round(r["bias_K"], 3),
                     None if kap is None else round(kap, 2)))
display(pd.DataFrame(rows, columns=["site", "model", "albedo", "RMSE (K)", "bias (K)", "kappa (1e-4 cm^2/s)"]))
for s in SITES_:
    h = fc[s]["holdout_deepest"]
    print(f"{s}: deepest sensor ({h['depth_cm']:.0f} cm) held out: K_d {h['kd_mW']:.2f}, "
          f"predicts it to {h['residual_K']:+.2f} K;  TG only {fc[s]['split_TG']['kd_mW']:.2f}, "
          f"TR only {fc[s]['split_TR']['kd_mW']:.2f}")

In [ ]:
chk = nb.Checks("Compared with the paper (Table 1; Section 4.2):")
paper = {"A15": dict(fit=("0.136", "0.90", "0.00", "0.80"), glob=("0.138", "0.90", "+0.02", "0.56"), ms=("0.91", "+0.14"), keff="9.8"),
         "A17": dict(fit=("0.137", "0.43", "0.00", "0.98"), glob=("0.143", "0.71", "+0.01", "0.56"), ms=("0.62", "+0.05"), keff="12.1")}
for s in SITES_:
    P = paper[s]
    for key, lab, (A, rmse, bias, kap) in (("fit_with_diffusivity", "joint fit", P["fit"]), ("global_3p4", "K_d=3.4, A refitted", P["glob"])):
        r = fc[s][key]
        chk.add(f"{s} {lab}: albedo", r["A"], A, ".3f")
        chk.add(f"{s} {lab}: RMSE (K)", r["rmse_K"], rmse, ".2f")
        chk.add(f"{s} {lab}: bias (K)", r["bias_K"], bias, ".2f" if bias == "0.00" else "+.2f")
        chk.add(f"{s} {lab}: kappa", sum(r["kappa_model"]) / 2, kap, ".2f")
    chk.add(f"{s} Martinez & Siegler: RMSE (K)", fc[s]["martinez_forward"]["rmse_K"], P["ms"][0], ".2f")
    chk.add(f"{s} Martinez & Siegler: bias (K)", fc[s]["martinez_forward"]["bias_K"], P["ms"][1], "+.2f")
    chk.add(f"{s} effective conductivity (mW/m/K)", fc[s]["effective_conductivity_mW"]["mean"], P["keff"], ".1f")
chk.show()

## 4. The surface against Diviner (SI Table S11, SI Figure S5)

The modeled day–night surface temperatures against the Diviner measurements.
Needs the Diviner tiles (notebook 00) to run; the script also makes SI Figure S5.

In [ ]:
DIVINER = ("data/diviner/gcp/global_cumul_avg_cyl_20n30n_002.tab", "data/diviner/gcp/global_cumul_avg_cyl_10n20n_002.tab")
nb.step("compute/compute_joint_diviner.py", minutes=3, writes="joint_diviner.json", needs=DIVINER)
jd = nb.load("joint_diviner.json")
rows = [(s, m, *(round(jd[s]["models"][m]["zonal"].get(k, float("nan")), 2) for k in ("bias_K", "rmse_K")))
        for s in SITES_ for m in jd[s]["models"]]
display(pd.DataFrame(rows, columns=["site", "model", "bias vs Diviner (K)", "RMSE vs Diviner (K)"]))

## 5. The measured gradient under each window choice (Sections 3.1, 3.2)

In [ ]:
nb.step("compute/compute_gradient_epochs.py", minutes=0.2, writes="gradient_epochs.json")
ge = nb.load("gradient_epochs.json")["sites"]
for s in SITES_:
    print(f"{s}: model gradients (K/m)", {k: round(v, 2) for k, v in ge[s]["model_gradient_K_per_m"].items()})
    print(f"     observed (K/m)       ", {k: (round(v, 2) if isinstance(v, (int, float)) else
                                          round(v.get("gradient_K_per_m", float("nan")), 2) if isinstance(v, dict) else v)
                                      for k, v in ge[s]["observed"].items()})

## 6. The radiative coefficient $\chi$ and the density (Section 4.2, SI Table S14)

The joint fit redone for $\chi$ = 1.5–3.2 and with each site's own density.
Full run about 50 min; "quick" shows the stored result.

In [ ]:
nb.step("compute/compute_joint_chi_density.py", minutes=50, writes="joint_chi_density.json")
cd = nb.load("joint_chi_density.json")
cases = list(cd["sites"]["A15"])
display(pd.DataFrame({s: [round(cd["sites"][s][c]["kd_star_mW"], 2) for c in cases] for s in SITES_} |
                     {"A17 - A15": [round(cd["contrast_mW"][c], 2) for c in cases]}, index=cases)
        .rename_axis("K_d* (mW m^-1 K^-1)"))

In [ ]:
chk = nb.Checks("Compared with the paper (Table 3):")
P = {"A15": ("0.30", "<0.05", "<0.05", "0.40", "0.15", "<0.05", "0.52", "0.45"),
     "A17": ("0.33", "<0.05", "0.15", "0.51", "0.15", "<0.05", "0.64", "0.55")}
for s in SITES_:
    e, p = eb[s], P[s]
    for (lab, key), val in zip((("statistical", "sigma_stat"), ("solver", "sigma_solver"), ("Q_b", "sigma_Qb"),
                                ("density", "sigma_rho"), ("c_p", "sigma_cp"), ("K_s", "sigma_Ks")), p[:6]):
        chk.add(f"{s}: {lab}", e[key], val, ".2f")
    for key in ("sigma_zb", "sigma_thr", "sigma_window", "sigma_epoch"):
        chk.add(f"{s}: {key.split('_')[1]}", e[key], "<0.05")
    chk.add(f"{s}: total (quadrature)", e["total_quadrature"], p[6], ".2f")
    k = cd["sites"][s]
    chk.add(f"{s}: chi 2.2-3.2 (half range)", abs(k["chi_2.2"]["kd_star_mW"] - k["chi_3.2"]["kd_star_mW"]) / 2, p[7], ".2f")
    chk.add(f"{s}: H 3-10 cm", e["conditional"]["H"], "<0.05")
    chk.add(f"{s}: albedo form", max(abs(v - e["kd_star_mW"]) for v in e["conditional"]["albedo_form_kd_mW"].values()), "<0.05")
chk.show()

## 7. Likelihood-ratio tests (Tables 1, 2; Sections 3.2, 4.1)

The probability $p$ that the data would fit this much better than the
global $K_d=3.4$ (or than one shared $K_d$) by chance.

In [ ]:
nb.step("compute/compute_likelihood_ratio.py", minutes=0.2, writes="likelihood_ratio.json")
lr = nb.load("likelihood_ratio.json")
po = lr["pooled"]
display(pd.DataFrame([("per-site K_d", "—", 0.0, "—"),
                      ("one shared K_d", po["kd_shared_mW"], round(po["shared_vs_per_site"]["delta_J"], 2), f'{po["shared_vs_per_site"]["p"]:.2g}'),
                      ("global K_d = 3.4", 3.4, round(po["global_vs_per_site"]["delta_J"], 2), f'{po["global_vs_per_site"]["p"]:.2g}')],
                     columns=["model (Table 2)", "K_d", "Delta J", "p"]))

In [ ]:
chk = nb.Checks("Compared with the paper (Tables 1, 2; Section 3.2):")
sa = lr["standard_albedo_0p12"]
for s, (rm, bi, kap, p_pub, p_ref, kd012) in (("A15", ("1.77", "+1.53", "0.56", "6e-8", "9e-7", "4.8")),
                                              ("A17", ("2.09", "+1.97", "0.57", "5e-21", "1e-14", "5.6"))):
    h = sa[s]["hayne_as_published"]
    chk.add(f"{s} published Hayne (A=0.12, K_d=3.4): RMSE", h["sensor_rmse_K"], rm, ".2f")
    chk.add(f"{s} published Hayne: bias", h["sensor_bias_K"], bi, "+.2f")
    chk.add(f"{s} published Hayne: kappa", sum(h["kappa_model_1e4_cm2_s"]) / 2, kap, ".2f")
    chk.add(f"{s} published Hayne: p", sa[s]["global_3p4_at_standard_albedo"]["p"], p_pub, ".0e")
    chk.add(f"{s} K_d=3.4, A refitted: p", lr["sites"][s]["global_3p4_with_diffusivity"]["p"], p_ref, ".0e")
    chk.add(f"{s} K_d* with the albedo fixed at 0.12", sa[s]["kd_star_mW"], kd012, ".1f")
chk.add("Table 2: one shared K_d", po["kd_shared_mW"], "5.3", ".1f")
chk.add("Table 2: shared, Delta J", po["shared_vs_per_site"]["delta_J"], "5.3", ".1f")
chk.add("Table 2: shared, p", po["shared_vs_per_site"]["p"], "0.02", ".2f")
chk.add("Table 2: global, Delta J", po["global_vs_per_site"]["delta_J"], "83.7", ".1f")
chk.add("Table 2: global, p", po["global_vs_per_site"]["p"], "7e-19", ".0e")
chk.show()

## 8. Figures

In [ ]:
nb.figures("figures/make_joint_figures.py", makes=("fig_joint_thermal_profiles", "fig_joint_robustness"), minutes=2)
nb.figures("compute/compute_joint_diviner.py", makes="fig_joint_diviner_closure", minutes=3, needs=DIVINER)

**Figure 3.** Mean temperature profiles: the joint fit, the global value and
Martinez & Siegler against the sensors.

In [ ]:
nb.show("fig_joint_thermal_profiles")

**SI Figure S5.** The day–night surface temperature against Diviner.

In [ ]:
nb.show("fig_joint_diviner_closure")

**SI Figure S6.** $K_d$ under each changed input, grouped as in Table 3.

In [ ]:
nb.show("fig_joint_robustness")

Next: **05_annual_wave_and_posterior.ipynb**.